In [23]:


from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os
dataset_path = "/content/drive/MyDrive/placement_data.csv"
# os.listdir is used to list contents of a directory.
# If you intend to read the file, you would typically use a function like pandas.read_csv()
# or open() for general file reading.
# If you just want to check if the file exists, you can use os.path.exists()
print(f"File exists: {os.path.exists(dataset_path)}")
print(f"Path is a file: {os.path.isfile(dataset_path)}")
# If you want to list the directory containing the file, use:
# print(os.listdir(os.path.dirname(dataset_path)))

File exists: True
Path is a file: True


In [25]:
import pandas as pd
df = pd.read_csv(dataset_path)
df['dsa_cgpa_ratio']=df['DSA_Solved']/df['CGPA']

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Encode target + categorical columns
le = LabelEncoder()
df['Placement'] = le.fit_transform(df['Placement']) # Yes=1, No=0
df['Internship'] = le.fit_transform(df['Internship']) # Yes=1, No=0

# One-hot encode Branch
df = pd.get_dummies(df, columns=['Branch'], drop_first=True)

# 2. Split features + target
X = df.drop('Placement', axis=1)
y = df['Placement']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train 3 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred))
    print("-"*40)

# 4. Pick best model
best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name} with {results[best_model_name]*100:.2f}% accuracy")

Logistic Regression Accuracy: 95.00%
              precision    recall  f1-score   support

           0       0.97      0.89      0.93        38
           1       0.94      0.98      0.96        62

    accuracy                           0.95       100
   macro avg       0.95      0.94      0.95       100
weighted avg       0.95      0.95      0.95       100

----------------------------------------
Random Forest Accuracy: 89.00%
              precision    recall  f1-score   support

           0       0.85      0.87      0.86        38
           1       0.92      0.90      0.91        62

    accuracy                           0.89       100
   macro avg       0.88      0.89      0.88       100
weighted avg       0.89      0.89      0.89       100

----------------------------------------
XGBoost Accuracy: 91.00%
              precision    recall  f1-score   support

           0       0.85      0.92      0.89        38
           1       0.95      0.90      0.93        62

    acc

In [21]:
rf = models["Random Forest"]
importances = rf.feature_importances_
pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values('Importance', ascending=False)

,Feature,Importance
0,CGPA,0.503816
1,DSA_Solved,0.202055
4,dsa_cgpa_ratio,0.130258
2,Projects,0.097429
3,Internship,0.032424
8,Branch_ME,0.013615
6,Branch_EE,0.009397
7,Branch_IT,0.005730
5,Branch_ECE,0.005274


In [22]:
import joblib
joblib.dump(models[best_model_name], 'placement_model.pkl')

['placement_model.pkl']

In [27]:
!pip install streamlit
import streamlit as st
import pandas as pd
import joblib

model = joblib.load('placement_model.pkl')

st.title("🎓 Placement Predictor")
st.write("Check your placement probability based on CGPA, DSA, etc.")

cgpa = st.slider("CGPA", 6.0, 10.0, 8.0, 0.1)
dsa = st.slider("DSA Problems Solved", 0, 300, 150)
projects = st.slider("Projects Count", 0, 10, 3)
internship = 1 if st.selectbox("Internship", ["No", "Yes"]) == "Yes" else 0
branch = st.selectbox("Branch", ["CSE", "IT", "ECE", "ME", "CIVIL"])

dsa_cgpa_ratio = dsa / cgpa
branch_cols = {
    'Branch_IT': 1 if branch == 'IT' else 0,
    'Branch_ME': 1 if branch == 'ME' else 0,
    'Branch_ECE': 1 if branch == 'ECE' else 0,
    'Branch_CIVIL': 1 if branch == 'CIVIL' else 0
}

input_data = pd.DataFrame([[cgpa, dsa, projects, internship, dsa_cgpa_ratio,
                           branch_cols['Branch_ME'], branch_cols['Branch_ECE'],
                           branch_cols['Branch_IT'], branch_cols['Branch_CIVIL']]],
                         columns=['CGPA', 'DSA_Solved', 'Projects_count', 'Internship',
                                  'dsa_cgpa_ratio', 'Branch_ME', 'Branch_ECE',
                                  'Branch_IT', 'Branch_CIVIL'])

if st.button("Predict"):
    prob = model.predict_proba(input_data)[0][1] * 100
    st.metric("Placement Probability", f"{prob:.1f}%")

    if prob > 70:
        st.success("High chance ✅ Keep it up!")
    elif prob > 40:
        st.warning("Medium chance ⚠️ Focus on DSA/Internship")
    else:
        st.error("Low chance ❌ Improve CGPA + DSA")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.4 MB/s eta 0:00:00


2026-06-02 14:50:09.013 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.193 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-02 14:50:09.194 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.197 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.200 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.202 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 14:50:09.203 Thread 'MainThread': mi